## 3.6 Boosting 思想 - XGBoost 算法

#### 1. XGBoost是什么？
XGBoost（Extreme Gradient Boosting）是一个高效、灵活且可扩展的梯度提升框架，旨在提供更快的训练速度和更高的性能。<br>
XGBoost 在 Gradient Boosting 的基础上引入了正则化项、列抽样、并行计算等多种优化技术，使得它在处理大规模数据集和复杂模型时表现出色。<br>
XGBoost 在许多机器学习竞赛中表现优异，成为了数据科学家和机器学习工程师的首选工具之一。<br>

#### 2. XGBoost的核心思想，与 Gradient Boosting的区别：
1. 正则化：XGBoost 在损失函数中引入了正则化项，以控制模型的复杂度，防止过拟合。正则化项包括 L1 正则化（Lasso）和 L2 正则化（Ridge），可以帮助模型更好地泛化到未见过的数据。
2. 列抽样：XGBoost 支持在训练过程中进行列抽样，即在每次分裂节点时随机选择一部分特征进行分裂。这种机制可以增加模型的多样性，减少过拟合，并提高模型的性能。
3. 并行计算：XGBoost 通过利用多线程和分布式计算来加速模型的训练过程。它可以在多核 CPU 和 GPU 上高效地运行，大大缩短了训练时间，尤其是在处理大规模数据集时。
4. 处理缺失值：XGBoost 内置了处理缺失值的机制，可以自动识别和处理数据中的缺失值，无需进行额外的数据预处理。
5. 支持多种损失函数：XGBoost 支持多种损失函数，包括回归问题中的均方误差（MSE）和分类问题中的对数损失（Log Loss），使得它能够适应不同类型的任务。
6. 提供了丰富的参数调优选项：XGBoost 提供了大量的参数，可以通过调整这些参数来优化模型的性能，如学习率（learning_rate）、树的最大深度（max_depth）、子样本比例（subsample）等。

#### 3. XGBoost的推导过程

##### 3.1 目标函数的定义
XGBoost 的目标函数由两部分组成：损失函数和正则化项。损失函数用于衡量模型的预测误差，而正则化项用于控制模型的复杂度，防止过拟合。目标函数的定义如下：
$$
L(\phi) = \sum_{i=1}^{n} l(y_i, \hat{y}_i) + \sum_{k=1}^{K} \Omega(f_k)
$$
其中，$l(y_i, \hat{y}_i)$ 是损失函数，$\Omega(f_k)$ 是正则化项，$n$ 是样本数量，$K$ 是树的数量，$f_k$ 是第 $k$ 棵树的模型。
其中正则化项 $\Omega(f_k)$ 的定义如下：
$$
\Omega(f) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2
$$
其中，$T$ 是树的叶子节点数量，$w_j$ 是第 $j$ 个叶子节点的权重，$\gamma$ 和 $\lambda$ 是正则化参数，分别控制树的复杂度和叶子节点权重的大小。

##### 3.2 逐步加树
在第t轮中，我们需要训练一个新的树 $f_t$ 来拟合当前模型的残差。

##### 3.3 关键推导：对Loss函数进行二阶泰勒展开
由于Loss函数通常比较复杂，XGBoost采用二阶近似来优化目标函数。具体来说，在第t轮中，我们对目标函数进行二阶泰勒展开，得到如下近似形式：
$$
L^{(t)} \approx \sum_{i=1}^{n} [g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2] + \Omega(f_t)
$$
其中，$g_i$ 是第 $i$ 个样本的梯度（即残差），$h_i$ 是第 $i$ 个样本的二阶梯度（即残差的二阶导数）。通过优化这个近似的目标函数，我们可以有效地训练新的树 $f_t$ 来拟合当前模型的残差，从而逐步提升整体模型的性能。

##### 3.4 把数结构带入：按叶子节点聚合
在XGBoost中，每棵树的结构是由叶子节点组成的。为了优化目标函数，我们需要将树的结构带入到目标函数中进行优化。具体来说，我们可以将每个样本分配到对应的叶子节点上，并计算每个叶子节点的权重。<br>
假设第 $t$ 棵树有 $T$ 个叶子节点，我们可以将样本分配到对应的叶子节点上，并计算每个叶子节点的权重 $w_j$，其中 $j$ 是叶子节点的索引。<br>
通过将树的结构带入到目标函数中，我们可以得到如下的优化问题：
$$
\min_{w} \sum_{j=1}^{T} [G_j w_j + \frac{1}{2} H_j w_j^2] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2
$$
其中，$G_j$ 是第 $j$ 个叶子节点的梯度和，$H_j$ 是第 $j$ 个叶子节点的二阶梯度和。通过优化这个问题，我们可以得到每个叶子节点的权重 $w_j$，从而构建出新的树 $f_t$ 来拟合当前模型的残差。

##### 3.5 计算每个叶子节点的权重
对每个叶子节点求导并设置为零，我们可以得到每个叶子节点的权重 $w_j$ 的计算公式如下：
$$
w_j = -\frac{G_j}{H_j + \lambda}
$$
其中，$G_j$ 是第 $j$ 个叶子节点的梯度和，$H_j$ 是第 $j$ 个叶子节点的二阶梯度和，$\lambda$ 是正则化参数。通过计算每个叶子节点的权重，我们可以构建出新的树 $f_t$ 来拟合当前模型的残差，从而逐步提升整体模型的性能。

##### 3.6 最优目标值 - 用于评估树好不好
通过将每个叶子节点的权重带入到目标函数中，我们可以计算出当前树的最优目标值（即最小化后的目标函数值），用于评估树的好坏。具体来说，最优目标值的计算公式如下：
$$
L^{(t)} = -\frac{1}{2} \sum_{j=1}^{T} \frac{G_j^2}{H_j + \lambda} + \gamma T
$$
其中，$G_j$ 是第 $j$ 个叶子节点的梯度和，$H_j$ 是第 $j$ 个叶子节点的二阶梯度和，$\lambda$ 是正则化参数，$\gamma$ 是控制树的复杂度的正则化参数。通过计算最优目标值，我们可以评估当前树的好坏，从而选择最优的树来构建模型，逐步提升整体模型的性能。

##### 3.6 分裂增益 Gain
在XGBoost中，分裂增益（Gain）用于评估每次分裂的效果。具体来说，分裂增益的计算公式如下：
$$
Gain = \frac{1}{2} \left( \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right) - \gamma
$$
其中，$G_L$ 和 $H_L$ 分别是左子树的梯度和和二阶梯度和，$G_R$ 和 $H_R$ 分别是右子树的梯度和和二阶梯度和，$\lambda$ 是正则化参数，$\gamma$ 是控制树的复杂度的正则化参数。通过计算分裂增益，我们可以评估每次分裂的效果，从而选择最优的分裂方式来构建树。